<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_11_importing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 📘 **Module 11 — Importing Data Sets** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 11 — Importing Data Sets
*IBM Data Analysis with Python · Module 1 of 6 (Module 11 of 16)*

The data analysis track starts here. Every analysis begins the same way: **load the data, look at it, write down what's broken**. This module is about the first two steps.

We'll use a **shared dataset** across modules 11–16: the classic *auto-mpg* dataset (1970s–80s cars: mpg, weight, horsepower, etc.). Every later module reuses it, so the steps connect end-to-end.

### What you'll cover
1. Where datasets come from (CSV, Excel, JSON, SQL, URL)
2. `pd.read_csv` and its 10 most useful options
3. Reading Excel and JSON
4. Reading from a SQLite database
5. The "first inspection" toolkit — `shape`, `dtypes`, `head`, `info`, `describe`
6. Identifying features vs the target variable
7. Practice


In [ ]:
!pip -q install pandas numpy openpyxl
import pandas as pd, numpy as np

## 1. Where data lives — five sources

| Source | Tool |
|---|---|
| Local CSV | `pd.read_csv(path)` |
| Excel | `pd.read_excel(path)` |
| JSON | `pd.read_json(path)` |
| SQL database | `pd.read_sql(query, conn)` |
| URL (csv/zip/parquet) | `pd.read_csv(url)` — works the same |

Pandas hides the fact that it's reading from disk vs HTTP. A URL is just a "path".

## 2. The shared dataset — auto-mpg

In [ ]:
url  = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
cols = ["mpg","cylinders","displacement","horsepower","weight",
        "acceleration","model_year","origin","car_name"]
cars = pd.read_csv(url, sep=r"\s+", names=cols, na_values="?")
print(cars.shape)
cars.head()

### `pd.read_csv` — the 10 options worth knowing

| Option | Meaning |
|---|---|
| `sep=` | column separator (default `,`; tab is `\t`) |
| `names=` | provide column names if missing |
| `header=` | row number of header (default 0; `None` if none) |
| `na_values=` | extra strings to treat as NaN (`"?"`, `"NA"`, ...) |
| `dtype=` | force dtypes (`{"id": int, "amount": float}`) |
| `parse_dates=` | parse columns as datetime |
| `index_col=` | which column becomes the index |
| `usecols=` | only read these columns |
| `nrows=` | only read the first N rows (for big files) |
| `chunksize=` | iterate over the file in chunks (huge files) |

In [ ]:
# Just read 5 rows + selected columns — useful for inspecting a huge file
small = pd.read_csv(url, sep=r"\s+", names=cols, na_values="?",
                    usecols=["mpg","weight","horsepower"], nrows=5)
print(small)

## 3. Reading Excel and JSON

In [ ]:
# Make and read a small Excel file
import pathlib, tempfile
TMP = pathlib.Path(tempfile.gettempdir())
xlsx = TMP / "demo.xlsx"
cars.head(20).to_excel(xlsx, sheet_name="cars", index=False)

back = pd.read_excel(xlsx, sheet_name="cars")
print(back.head())

In [ ]:
# JSON: one record per dict (records orient is the most natural)
jpath = TMP / "cars.json"
cars.head(5).to_json(jpath, orient="records", indent=2)
print(jpath.read_text()[:300])
print(pd.read_json(jpath, orient="records"))

## 4. Reading from a SQLite database

Most production data lives in databases. The pattern: open a connection, hand a query to `pd.read_sql`.

In [ ]:
import sqlite3
db = TMP / "cars.db"
con = sqlite3.connect(db)
cars.to_sql("cars", con, if_exists="replace", index=False)

# Now query it back like SQL
q = """SELECT origin, AVG(mpg) AS avg_mpg, COUNT(*) AS n
       FROM cars
       GROUP BY origin
       ORDER BY avg_mpg DESC"""
result = pd.read_sql(q, con)
print(result)
con.close()

## 5. The first-inspection toolkit

Five lines you should run on **every** new dataset, in order. Each answers one specific question.

In [ ]:
print("shape:", cars.shape)            # how big?

In [ ]:
cars.head()                          # what does a row look like?

In [ ]:
cars.dtypes                          # are types right?

In [ ]:
cars.info()                          # how many missing per column?

In [ ]:
cars.describe(include="all").T       # what are the ranges + categories?

## 6. Features vs target — naming what you have

In supervised learning, columns split into:
- **Target** (a.k.a. label, dependent variable, `y`) — the thing you want to predict.
- **Features** (a.k.a. independent variables, `X`) — everything you'll use to predict it.

For auto-mpg the target is **mpg** (we want to predict fuel efficiency). Everything else is a candidate feature.

In [ ]:
# Make this explicit — it carries through the next 5 modules
target   = "mpg"
features = ["cylinders","displacement","horsepower","weight","acceleration","model_year","origin"]

print("target  :", target)
print("features:", features)
print("rows×cols of X:", cars[features].shape, "  y:", cars[target].shape)

## 7. Practice

1. Use `pd.read_csv` to load the famous Iris dataset from
   `https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data` (no header). Use the column names `["sepal_len","sepal_wid","petal_len","petal_wid","class"]`. Print the shape and the first 5 rows.
2. Save your auto-mpg cars dataframe to a SQLite db, then write a SQL query that returns the **5 most-fuel-efficient cars** by `mpg`.
3. Read just the columns `["mpg","horsepower","weight"]` from the auto-mpg URL with `usecols=`. Then print the row with the highest `weight`.


In [ ]:
# 1)
iris = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data",
    header=None, names=["sepal_len","sepal_wid","petal_len","petal_wid","class"])
print(iris.shape); print(iris.head())

# 2)
import sqlite3, tempfile, pathlib
db = pathlib.Path(tempfile.gettempdir()) / "cars2.db"
con = sqlite3.connect(db); cars.to_sql("cars", con, if_exists="replace", index=False)
print(pd.read_sql("SELECT car_name, mpg FROM cars ORDER BY mpg DESC LIMIT 5", con))
con.close()

# 3)
small = pd.read_csv(url, sep=r"\s+", names=cols, na_values="?",
                    usecols=["mpg","horsepower","weight"])
print(small.iloc[small["weight"].idxmax()])

## Recap

✅ Load CSV, Excel, JSON, SQL — same `read_*` family
✅ Use `pd.read_csv`'s 10 options (sep, names, na_values, dtype, parse_dates, index_col, usecols, nrows, chunksize, header)
✅ Run the 5-line first-inspection ritual on any new dataset
✅ Name what's the target and what are the features

### Next
**Module 12 — Data Wrangling** (cleaning the things this module just helped you find).